In [ ]:
# Set up the pinned official V-JEPA source and check the GPU.
import sys, subprocess
from pathlib import Path
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'timm==1.0.26', 'einops', 'pillow'], check=True)
upstream = Path('/content/vjepa2')
if not upstream.exists():
    subprocess.run(['git', 'clone', 'https://github.com/facebookresearch/vjepa2.git', str(upstream)], check=True)
subprocess.run(['git', '-C', str(upstream), 'checkout', '204698b45b3712590f06245fbfba32d3be539812'], check=True)
import torch
assert torch.cuda.is_available(), 'Select a GPU runtime'
print('Python:', sys.version)
print('PyTorch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0))
print('GPU memory GiB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))


In [ ]:
# Load frozen pretrained ViT-L and verify one clip fits.
import gc, time
sys.path.insert(0, str(upstream))
from src.hub import backbones
backbones.VJEPA_BASE_URL = 'https://dl.fbaipublicfiles.com/vjepa2'
encoder, predictor = backbones.vjepa2_1_vit_large_384(pretrained=True)
del predictor
gc.collect()
encoder = encoder.eval().requires_grad_(False).to('cuda')
torch.cuda.reset_peak_memory_stats()
t0 = time.perf_counter()
with torch.inference_mode(), torch.autocast('cuda', dtype=torch.float16):
    smoke = encoder(torch.zeros(1, 3, 16, 384, 384, device='cuda'))
assert tuple(smoke.shape) == (1, 4608, 1024), smoke.shape
assert torch.isfinite(smoke).all()
print('Encoder output:', tuple(smoke.shape))
print('Peak GPU GiB:', round(torch.cuda.max_memory_allocated()/1024**3, 3))
print('Smoke seconds:', round(time.perf_counter()-t0, 2))
del smoke


In [ ]:
# Run the pinned experiment. Ground truth after initialization is for scoring only.
import contextlib, json
project = Path('/content/vjepa-poc')
if not project.exists():
    subprocess.run(['git', 'clone', '--branch', 'experiment/part-consistency-colab', 'https://github.com/skbdps/vjepa-poc.git', str(project)], check=True)
subprocess.run(['git', '-C', str(project), 'checkout', '461dd0264dfe023bc9639479a388fa2eda785692'], check=True)
sys.path.insert(0, str(project / 'experiments/day3'))
import part_consistency, run_experiment
part_consistency.self_test('/content/day3_self_test')
run_dir = Path('/content/day3_run')
run_dir.mkdir(exist_ok=True)
print('Running 3 calibration and 6 held-out clips, two encoder windows each...')
with (run_dir / 'run.log').open('w') as log, contextlib.redirect_stdout(log):
    report = run_experiment.run(run_dir, upstream, encoder=encoder)
print('EXPERIMENT_COMPLETE')
print(json.dumps({k: v['overall'] for k, v in report['methods'].items()}, indent=2))
print((run_dir / 'manifest.json').read_text())


In [ ]:
# Results: localization is measured at patch/tubelet resolution.
import pandas as pd
from IPython.display import display, HTML, Video
rows = []
for method, values in report['methods'].items():
    m = values['overall']
    def ratio(metric):
        z = m[metric]
        return f"{z['numerator']}/{z['denominator']}"
    rows.append({'Method': method, 'Correct visible': ratio('localization_accuracy_given_visible'), 'Accuracy %': round(100*m['localization_accuracy_given_visible']['rate'],1), 'False presence when hidden': ratio('false_presence_given_absent'), 'Recovered after occlusion': ratio('recovery_accuracy'), 'Second-window accuracy %': round(100*values['by_window']['across_window']['localization_accuracy_given_visible']['rate'],1)})
display(HTML('<h2>Day 3: part consistency results</h2><p>Frozen V-JEPA 2.1 ViT-L on a Tesla T4. 3 calibration clips, 6 held-out synthetic clips. Four tagged parts per clip; 315 visible checks, 36 absent checks, 4 recovery events. Nine ambiguous checks excluded.</p>'))
display(pd.DataFrame(rows).set_index('Method'))
display(HTML('<p><b>Interpretation:</b> V-JEPA ties optical flow on visible localization in this small diagnostic, with fewer false-presence errors and 2/4 recoveries. Cars stay in separate lanes. This does not establish camera-angle robustness, face consistency or generative editing. Both-frame annotations initialize V-JEPA; optical flow receives frame 0. Features are offline within each 16-frame window.</p>'))
for video in sorted((run_dir / 'results').glob('*comparison.mp4')):
    print(video.name)
    display(Video(str(video), embed=True, width=640))


In [ ]:
# Preserve all results and reusable feature caches, excluding model weights.
import shutil
from google.colab import files
manifest_path = run_dir / 'manifest.json'
manifest = json.loads(manifest_path.read_text())
manifest['experiment_commit'] = '461dd0264dfe023bc9639479a388fa2eda785692'
manifest['run_date_IST'] = '2026-10-09'
manifest_path.write_text(json.dumps(manifest, indent=2))
shutil.copy2('/content/day3_self_test/self_test.json', run_dir / 'evaluator_self_test.json')
archive = shutil.make_archive('/content/VJEPA_Day3_Run_2026-10-09', 'zip', run_dir)
print('Completed archive:', Path(archive).name, round(Path(archive).stat().st_size/1024**2, 1), 'MiB')
files.download(archive)


In [ ]:
# Install the pinned official SAM 2.1 source for real-video part masks.
import os, urllib.request
sam_root = Path('/content/sam2')
if not sam_root.exists():
    subprocess.run(['git', 'clone', 'https://github.com/facebookresearch/sam2.git', str(sam_root)], check=True)
subprocess.run(['git', '-C', str(sam_root), 'checkout', '2b90b9f5ceec907a1c18123530e92e794ad901a4'], check=True)
sam_env = os.environ.copy()
sam_env['SAM2_BUILD_CUDA'] = '0'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(sam_root), '--no-build-isolation'], env=sam_env, check=True)
sam_ckpt = Path('/content/sam2.1_hiera_tiny.pt')
if not sam_ckpt.exists():
    urllib.request.urlretrieve('https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_tiny.pt', sam_ckpt)
print('SAM2_SETUP_COMPLETE', sam_ckpt.stat().st_size, 'bytes')


In [ ]:
# Day 4: calibration + development only; final test remains sealed.
import importlib.util
subprocess.run(['git', '-C', str(project), 'fetch', 'origin', 'experiment/part-consistency-colab'], check=True)
subprocess.run(['git', '-C', str(project), 'checkout', '5eb60adad99d5c7fb1b566e3d6ef39664bf8f69c'], check=True)
day4_src = project / 'experiments/day4'
sys.path.insert(0, str(day4_src))
spec = importlib.util.spec_from_file_location('day4_experiment', day4_src / 'run_experiment.py')
day4 = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = day4
spec.loader.exec_module(day4)
print('Tracking checks:', day4.tracking.self_test())
print('Benchmark checks:', day4.bench.self_test())
day4_out = Path('/content/day4_run')
day4_development, day4_frozen = day4.run_development(day4_out, upstream, encoder=encoder)
print('FROZEN_CONFIG_READY', day4_frozen['selected_method'])


In [ ]:
# Preserve the development freeze, then run the independent real-car demo.
import zipfile, hashlib
dev_bundle = Path('/content/VJEPA_Day4_Development_Freeze.zip')
with zipfile.ZipFile(dev_bundle, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    for p in day4_out.rglob('*'):
        if p.is_file() and 'features' not in p.relative_to(day4_out).parts:
            z.write(p, str(p.relative_to(day4_out)))
files.download(str(dev_bundle))
subprocess.run(['git', '-C', str(project), 'fetch', 'origin', 'experiment/part-consistency-colab'], check=True)
subprocess.run(['git', '-C', str(project), 'checkout', '9243a3cd459cc79a2d86122fea8679ee5e5e3478'], check=True)
assert day4.source_digest() == day4_frozen['source_digest']
real_root = Path('/content/day4_real/car-roundabout')
real_root.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile('/content/DAVIS_Car_Roundabout_64_Source.zip') as z:
    z.extractall(real_root)
source_manifest = json.loads((day4_src / 'car_roundabout_source_manifest.json').read_text())
for row in source_manifest['frames']:
    p = real_root / 'frames' / f"{row['frame_index']:05d}.jpg"
    assert hashlib.sha256(p.read_bytes()).hexdigest() == row['sha256']
sys.path.insert(0, str(sam_root))
import real_video, part_editor
print('Edit checks:', part_editor.self_test())
real_out = Path('/content/day4_real/results')
real_results, real_timing = real_video.run(real_root / 'frames', day4_src / 'car_roundabout_annotations.json', real_out, checkpoint=str(sam_ckpt))
print('REAL_PART_TRACKING_COMPLETE', json.dumps(real_timing, indent=2))
display(Video(str(real_out / 'door_recolor.mp4'), embed=True, width=854))


In [ ]:
# Archive the real run with independent annotations and attribution.
real_bundle = Path('/content/VJEPA_Day4_Real_Car_Roundabout.zip')
with zipfile.ZipFile(real_bundle, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    for p in real_out.rglob('*'):
        if p.is_file():
            z.write(p, 'results/' + str(p.relative_to(real_out)))
    for name in ['car_roundabout_annotations.json', 'car_roundabout_source_manifest.json', 'ATTRIBUTION.md']:
        z.write(day4_src / name, name)
print('REAL_ARCHIVE_READY', real_bundle.stat().st_size)
files.download(str(real_bundle))


In [ ]:
# Final synthetic test: configuration was committed before these results.
subprocess.run(['git', '-C', str(project), 'fetch', 'origin', 'experiment/part-consistency-colab'], check=True)
subprocess.run(['git', '-C', str(project), 'checkout', 'd6ad0e0c16ded18a5bb824aeecd33e727e34ff3c'], check=True)
committed_freeze = json.loads((day4_src / 'run_2026-10-09/frozen_config.json').read_text())
assert committed_freeze == json.loads((day4_out / 'frozen_config.json').read_text())
assert day4.source_digest() == committed_freeze['source_digest']
day4_test = day4.run_test(day4_out, upstream, encoder=encoder)
print('SEALED_TEST_FINISHED')


In [ ]:
# Replicate unchanged mask trackers on a second real car video.
subprocess.run(['git', '-C', str(project), 'fetch', 'origin', 'experiment/part-consistency-colab'], check=True)
subprocess.run(['git', '-C', str(project), 'checkout', 'dcdefecdd4bd06adb90635adfc2b14d1e393c46b'], check=True)
import importlib
importlib.reload(real_video)
shadow_base = Path('/content/day4_shadow')
shadow_base.mkdir(exist_ok=True)
with zipfile.ZipFile('/content/VJEPA_Car_Shadow_Inputs.zip') as z:
    z.extractall(shadow_base)
shadow_root = shadow_base / 'car-shadow'
shadow_source = json.loads((day4_src / 'car_shadow_source_manifest.json').read_text())
for row in shadow_source['frames']:
    p = shadow_root / 'frames' / f"{row['frame_index']:05d}.jpg"
    assert hashlib.sha256(p.read_bytes()).hexdigest() == row['sha256']
shadow_out = shadow_base / 'results'
shadow_results, shadow_timing = real_video.run(shadow_root / 'frames', day4_src / 'car_shadow_annotations.json', shadow_out, checkpoint=str(sam_ckpt))
print('SECOND_REAL_VIDEO_COMPLETE', json.dumps(shadow_timing, indent=2))
display(Video(str(shadow_out / 'door_recolor.mp4'), embed=True, width=854))


In [ ]:
# Calibrate a fixed specialist tracker; no SAM2 test clips run in this cell.
import sam2_benchmark
print('SAM2 readout checks:', sam2_benchmark.self_test())
sam2_out = Path('/content/day4_sam2_run')
sam2_calibration = sam2_benchmark.run_stage(sam2_out, 'calibration', checkpoint=str(sam_ckpt))
combined_archive = Path('/content/VJEPA_Day4_Test_Shadow_SAM2_Freeze.zip')
with zipfile.ZipFile(combined_archive, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    for root, prefix in [(day4_out, 'synthetic_vjepa'), (shadow_out, 'car_shadow'), (sam2_out, 'sam2_calibration')]:
        for p in root.rglob('*'):
            if p.is_file() and not any(x in ('features', 'frames') for x in p.relative_to(root).parts):
                z.write(p, prefix + '/' + str(p.relative_to(root)))
print('CALIBRATION_AND_RUN_ARCHIVE_READY', combined_archive.stat().st_size)
files.download(str(combined_archive))


In [ ]:
# Export completed results and the SAM2 freeze before starting its test.
files.download(str(combined_archive))
print('SAM2_FROZEN_CONFIG', (sam2_out/'sam2_frozen_config.json').read_text())


In [ ]:
# SAM2 held-out test. Calibration freeze committed before execution.
subprocess.run(['git','-C',str(project),'fetch','origin','experiment/part-consistency-colab'], check=True)
subprocess.run(['git','-C',str(project),'checkout','f9e2947b94f50bfd610e3854f12132491d2940a8'], check=True)
sam2_committed_freeze=json.loads((day4_src/'run_2026-10-09/sam2_frozen_config.json').read_text())
assert sam2_committed_freeze == json.loads((sam2_out/'sam2_frozen_config.json').read_text())
assert sam2_benchmark.source_digest() == sam2_committed_freeze['source_digest']
sam2_test=sam2_benchmark.run_stage(sam2_out, 'test', checkpoint=str(sam_ckpt))
print('SAM2_HELD_OUT_TEST_FINISHED')


In [ ]:
# Preserve all specialist predictions and scores, excluding reproducible input JPEGs.
sam2_archive=Path('/content/VJEPA_Day4_SAM2_Complete_Test.zip')
with zipfile.ZipFile(sam2_archive,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for p in sam2_out.rglob('*'):
        if p.is_file() and 'frames' not in p.relative_to(sam2_out).parts:
            z.write(p,str(p.relative_to(sam2_out)))
print('SAM2_COMPLETE_ARCHIVE_READY',sam2_archive.stat().st_size)
files.download(str(sam2_archive))


In [ ]:
# Completed results and two independent part editors; no inference required.
from pathlib import Path
import subprocess, json
import pandas as pd
from IPython.display import display, HTML
project=Path('/content/vjepa-poc')
if not project.exists():
    subprocess.run(['git','clone','--branch','experiment/part-consistency-colab','https://github.com/skbdps/vjepa-poc.git',str(project)],check=True)
subprocess.run(['git','-C',str(project),'fetch','origin','experiment/part-consistency-colab'],check=True)
subprocess.run(['git','-C',str(project),'checkout','66a91f969f583a245cf3e9f66977f24ae940bb55'],check=True)
run=project/'experiments/day4/run_2026-10-09'
sam=json.loads((run/'sam2_test/results.json').read_text())
vje=json.loads((run/'synthetic_test/results.json').read_text())
rows=[]
for name,m in [('SAM2.1 Tiny',sam['methods']['sam2_1_tiny']['overall']),('Selected VJEPA',vje['methods']['no_memory']['overall']),('Template + flow',vje['methods']['template_flow']['overall'])]:
    row={'Method':name}
    for title,key in [('Correct visible part','localization_accuracy_given_visible'),('False presence while hidden','false_presence_given_absent'),('Immediate recovery','recovery_accuracy')]:
        x=m[key]
        row[title]=f"{x['numerator']}/{x['denominator']} ({100*x['rate']:.1f}%)"
    rows.append(row)
display(HTML('<h2>Persistent parts: completed experiment</h2><p>18 held-out synthetic clips, one initial annotation per part. SAM2 is the strongest tested visible localizer. Occlusion recovery remains limited. The table scores two-frame patches; raw dense SAM2 masks separately report 60/539 hidden part-frames and 94.42% mean visible IoU.</p>'+pd.DataFrame(rows).to_html(index=False)+'<p>Real-video mask IoU: 86.18% on car-roundabout and 81.48% on car-shadow, against sparse approximate assistant-authored polygons. This is deterministic part recoloring; faces and generative editing remain untested. Full results, failure videos, caches and the audit are saved in GitHub.</p>'))
for clip in ['car_roundabout','car_shadow']:
    display(HTML((run/clip/'interactive_part_editor.html').read_text()))


In [ ]:
# Day 5: recover pinned SAM2 on the fresh T4 runtime.
import sys, subprocess, hashlib
from pathlib import Path
import torch
assert torch.cuda.is_available(), 'A GPU runtime is required'
print('GPU:',torch.cuda.get_device_name(0),'PyTorch:',torch.__version__)
project=Path('/content/vjepa-poc')
if not project.exists():
    subprocess.run(['git','clone','--branch','experiment/part-consistency-colab','https://github.com/skbdps/vjepa-poc.git',str(project)],check=True)
subprocess.run(['git','-C',str(project),'fetch','origin','experiment/part-consistency-colab'],check=True)
subprocess.run(['git','-C',str(project),'checkout','e2d549c4cf7dde887eb1153cbe612ebd97da3dc3'],check=True)
sys.path.insert(0,str(project/'experiments/day4'))
import real_video
sam_ckpt=real_video.prepare_sam2()
assert real_video.sha256_file(sam_ckpt)=='7402e0d864fa82708a20fbd15bc84245c2f26dff0eb43a4b5b93452deb34be69'
print('DAY5_GPU_READY',sam_ckpt)


In [ ]:
# Improved saved-track editor: paste a recipe when a file picker is unavailable.
from IPython.display import display, HTML
subprocess.run(['git','-C',str(project),'fetch','origin','experiment/part-consistency-colab'],check=True)
subprocess.run(['git','-C',str(project),'checkout','d153282d2c9a311638c0bb522c602dc77dd51688'],check=True)
for clip in ['car_roundabout','car_shadow']:
    display(HTML((project/'experiments/day4/run_2026-10-09'/clip/'interactive_part_editor.html').read_text()))


In [ ]:
# Day 5: fixed whole-car containment experiment. Source frozen before all GPU predictions.
import sys, subprocess
from pathlib import Path
project = Path('/content/vjepa-poc')
freeze_commit = 'de419ad916e57e04271c14fd830a8a52022b764e'
subprocess.run(['git','-C',str(project),'fetch','origin','experiment/part-consistency-colab'],check=True)
subprocess.run(['git','-C',str(project),'checkout',freeze_commit],check=True)
runner = project/'experiments/day5/run_parent_experiment.py'
frozen = project/'experiments/day5/frozen_config.json'
subprocess.run([sys.executable,'-u',str(runner),'--stage','smoke','--out','/content/day5_parent_run','--checkpoint','/content/checkpoints/sam2.1_hiera_tiny.pt','--frozen-path',str(frozen)],check=True)


In [ ]:
# Execute ALL 12 fixed test clips after successful smoke mechanics; do not tune on either split.
import json
smoke_report = json.loads(Path('/content/day5_parent_run/smoke/results.json').read_text())
assert len(smoke_report['clips']) == 3
assert smoke_report['frozen_config'] == json.loads(frozen.read_text())
print('Smoke completed; starting the unchanged 12-clip test.', flush=True)
cmd = [sys.executable,'-u',str(runner),'--stage','test','--out','/content/day5_parent_run','--checkpoint','/content/checkpoints/sam2.1_hiera_tiny.pt','--frozen-path',str(frozen)]
with Path('/content/day5_parent_run/test_execution.log').open('w') as log:
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        log.write(line); log.flush()
        if line.startswith('DAY5_'):
            print(line.rstrip(), flush=True)
    result = process.wait()
    if result:
        print(Path('/content/day5_parent_run/test_execution.log').read_text()[-6000:])
        raise subprocess.CalledProcessError(result, cmd)
print('DAY5_FULL_TEST_FINISHED', flush=True)


In [ ]:
# Preserve the completed fixed test before interpreting or extending it.
import json, zipfile, shutil, hashlib
from datetime import datetime, timezone
from IPython.display import display, HTML, Video
run_root = Path('/content/day5_parent_run')
report = json.loads((run_root/'test/results.json').read_text())
assert len(report['clips']) == 12
assert all(report['arms'][a][r]['overall']['scored_part_frames']==3024 for a in report['arms'] for r in report['arms'][a])
assert report['frozen_config'] == json.loads(frozen.read_text())
shutil.copy2(frozen,run_root/'frozen_config.json')
shutil.copy2(project/'experiments/day5/PROTOCOL.md',run_root/'PROTOCOL.md')
for relative, expected in report['frozen_config']['source_hashes'].items():
    source=project/'experiments'/relative
    assert hashlib.sha256(source.read_bytes()).hexdigest()==expected
    target=run_root/'source'/relative
    target.parent.mkdir(parents=True,exist_ok=True)
    shutil.copy2(source,target)
(run_root/'execution_manifest.json').write_text(json.dumps({'experiment_commit':freeze_commit,'completed_utc':datetime.now(timezone.utc).isoformat(),'test_clips':12,'smoke_clips':3,'runtime':report['runtime'],'no_parameter_selection':True},indent=2))
paths=[p for p in sorted(run_root.rglob('*')) if p.is_file() and 'frames' not in p.relative_to(run_root).parts and p.suffix not in ['.pt','.pth','.safetensors'] and p.name!='artifact_inventory.json']
inventory=[{'path':str(p.relative_to(run_root)),'bytes':p.stat().st_size,'sha256':hashlib.sha256(p.read_bytes()).hexdigest()} for p in paths]
(run_root/'artifact_inventory.json').write_text(json.dumps(inventory,indent=2))
archive=Path('/content/VJEPA_Day5_Parent_Constraint_Run.zip')
with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for p in paths+[run_root/'artifact_inventory.json']:
        z.write(p,str(p.relative_to(run_root)))
with zipfile.ZipFile(archive) as z:
    assert z.testzip() is None
print('DAY5_ARCHIVE_READY',archive.name,archive.stat().st_size,hashlib.sha256(archive.read_bytes()).hexdigest())
rows=[]
for arm in ['raw_part','parent_intersection']:
    s=report['arms'][arm]['effective_edit']['overall']
    rows.append('<tr><td>'+arm+'</td><td>'+format(s['wrong_car_pixels'],',')+'</td><td>'+format(100*s['mean_iou_given_visible']['rate'],'.2f')+'%</td><td>'+format(100*s['visible_micro_pixel_recall']['rate'],'.2f')+'%</td></tr>')
display(HTML('<h2>Day 5: independent parent-car constraint</h2><p>Completed all 12 fresh synthetic clips · 3,024 scored part-frames per arm. Same child predictions; extra independent whole-car tracks.</p><table><tr><th>Effective edit region</th><th>Wrong-car pixels</th><th>Visible mean IoU</th><th>Visible pixel recall</th></tr>'+''.join(rows)+'</table><p>Fixed criterion passed: <b>'+str(report['success']['passed'])+'</b>. This tests containment, not identity recovery. Whole-clip uncertainty and all four quality guards are in the saved report.</p>'))
display(Video(str(run_root/'test/test_crossing_6200_comparison.mp4'),embed=True,width=768))


In [ ]:
# Build controls from the COMPLETED run; this does not rerun or modify the tracker.
subprocess.run(['git','-C',str(project),'fetch','origin','experiment/part-consistency-colab'],check=True)
subprocess.run(['git','-C',str(project),'checkout','c55f436f9abce93b35b511e67f41e1140f271445'],check=True)
hierarchy = run_root/'hierarchy'
hierarchy_html = hierarchy/'interactive_hierarchy_editor.html'
subprocess.run([sys.executable,str(project/'experiments/day5/build_hierarchy_demo.py'),'--run-root',str(run_root),'--out',str(hierarchy_html)],check=True)
display(HTML(hierarchy_html.read_text()))
all_artifacts = Path('/content/VJEPA_Day5_All_Artifacts.zip')
with zipfile.ZipFile(all_artifacts,'w',compression=zipfile.ZIP_DEFLATED) as z:
    z.write(archive,archive.name)
    for p in sorted(hierarchy.rglob('*')):
        if p.is_file():
            z.write(p,'hierarchy/'+str(p.relative_to(hierarchy)))
print('DAY5_ALL_ARTIFACTS_READY',all_artifacts.name,all_artifacts.stat().st_size,hashlib.sha256(all_artifacts.read_bytes()).hexdigest())


In [ ]:
# Download the completed benchmark and controls; include runtime package versions.
from google.colab import files
import platform, PIL, cv2, numpy as np, torch
runtime_environment={'python':platform.python_version(),'torch':torch.__version__,'cuda':torch.version.cuda,'gpu':torch.cuda.get_device_name(0),'numpy':np.__version__,'pillow':PIL.__version__,'opencv':cv2.__version__,'experiment_commit':freeze_commit,'editor_helper_commit':'c55f436f9abce93b35b511e67f41e1140f271445'}
env_file=Path('/content/day5_runtime_environment.json')
env_file.write_text(json.dumps(runtime_environment,indent=2))
with zipfile.ZipFile(all_artifacts,'a',compression=zipfile.ZIP_DEFLATED) as z:
    if 'runtime_environment.json' not in z.namelist():
        z.write(env_file,'runtime_environment.json')
print('DAY5_DOWNLOAD',all_artifacts.name,all_artifacts.stat().st_size,hashlib.sha256(all_artifacts.read_bytes()).hexdigest())
files.download(str(all_artifacts))


In [ ]:
# Day 7: return to learned JEPA part identity; inspect available runtime.
from pathlib import Path
import sys, subprocess, json, hashlib, torch
print('DAY7_GPU',torch.cuda.get_device_name(),torch.__version__)
print('UPSTREAM_PRESENT',Path('/content/vjepa2/.git').exists())
print('CACHED_JEPA_WEIGHTS',[(p.name,p.stat().st_size) for p in Path('/root/.cache/torch/hub/checkpoints').glob('*jepa*')])
print('PROJECT',subprocess.check_output(['git','-C','/content/vjepa-poc','rev-parse','HEAD'],text=True).strip())


In [ ]:
# Official pinned V-JEPA backbone; the new persistent-part predictor is trained separately.
subprocess.run([sys.executable,'-m','pip','install','-q','timm==1.0.26','einops','pillow'],check=True)
upstream=Path('/content/vjepa2')
if not upstream.exists():
    subprocess.run(['git','clone','https://github.com/facebookresearch/vjepa2.git',str(upstream)],check=True)
subprocess.run(['git','-C',str(upstream),'checkout','204698b45b3712590f06245fbfba32d3be539812'],check=True)
print('DAY7_BACKBONE_SOURCE_READY')


In [ ]:
# Download/verify the official encoder once; no training or held-out scene is opened.
import gc, time
sys.path.insert(0,'/content/vjepa-poc/experiments/day4')
import run_experiment as day4_encoder
backbone_started=time.perf_counter()
encoder7=day4_encoder.load_encoder(upstream)
print('DAY7_ENCODER_LOADED',sum(p.numel() for p in encoder7.parameters()),round(time.perf_counter()-backbone_started,2))
del encoder7
gc.collect()
torch.cuda.empty_cache()
print('DAY7_BACKBONE_CACHED',[(p.name,p.stat().st_size) for p in (Path(torch.hub.get_dir())/'checkpoints').glob('*') if p.is_file()])


In [ ]:
# Frozen Day7 implementation and TRAIN/DEV extraction only.
day7_commit='1d9f12a5f51b061e33a4417d2789c203b5d1dc86'
project=Path('/content/vjepa-poc')
subprocess.run(['git','-C',str(project),'fetch','origin',day7_commit],check=True)
subprocess.run(['git','-C',str(project),'checkout','--detach',day7_commit],check=True)
day7_root=Path('/content/day7_jepa_run')
day7_features=day7_root/'features'
day7_root.mkdir(exist_ok=True)
day7_log=day7_root/'extraction_train_dev.log'
print('DAY7_EXTRACTION_FROZEN',day7_commit,flush=True)
cmd=[sys.executable,'-u',str(project/'experiments/day7/extract.py'),'--out',str(day7_features),'--upstream',str(upstream),'--splits','train','dev']
with day7_log.open('w') as log:
    process7=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    for line in process7.stdout:
        print(line,end='',flush=True)
        log.write(line);log.flush()
    returncode7=process7.wait()
assert returncode7==0, f'Extraction failed: {returncode7}'
print('DAY7_TRAIN_DEV_READY',len(json.loads((day7_features/'manifest.json').read_text())['clips']))


In [ ]:
# Train all three arms x three seeds; development selection only. No test access.
import os
assert returncode7==0
assert len(json.loads((day7_features/'manifest.json').read_text())['clips'])==45
assert subprocess.check_output(['git','-C',str(project),'rev-parse','HEAD'],text=True).strip()==day7_commit
train_env=dict(os.environ,OMP_NUM_THREADS='2',MKL_NUM_THREADS='2')
subprocess.run([sys.executable,str(project/'experiments/day7/train.py'),'--self-test'],check=True,env=train_env)
subprocess.run([sys.executable,'-c',"import sys,json;sys.path.insert(0,'/content/vjepa-poc/experiments/day7');from model import self_test;print(json.dumps(self_test('cuda')))"],check=True,env=train_env)
train_log7=day7_root/'training.log'
cmd=[sys.executable,'-u',str(project/'experiments/day7/train.py'),'--stage','train','--features',str(day7_features),'--out',str(day7_root)]
with train_log7.open('w') as log:
    train_process7=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,env=train_env)
    for line in train_process7.stdout:
        print(line,end='',flush=True)
        log.write(line);log.flush()
    train_returncode7=train_process7.wait()
assert train_returncode7==0, f'Training failed: {train_returncode7}'
print('DAY7_TRAINED_CHECKPOINTS_READY',hashlib.sha256((day7_root/'checkpoint_freeze.json').read_bytes()).hexdigest())
print('STOP: commit the checkpoint freeze before extracting any test features.')


In [ ]:
# Export the completed DEVELOPMENT freeze; this cell never opens test clips.
from google.colab import files
import zipfile, hashlib, json
from pathlib import Path
assert train_returncode7 == 0
freeze_path7 = day7_root / 'checkpoint_freeze.json'
freeze7 = json.loads(freeze_path7.read_text())
assert freeze7['test_accessed'] is False
assert len(freeze7['models']) == 9
assert not (day7_root / 'test').exists()
dev_zip7 = Path('/content/Day7_Development_Freeze.zip')
with zipfile.ZipFile(dev_zip7, 'w', zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(day7_root.rglob('*')):
        if path.is_file() and path.suffix in {'.json', '.csv', '.log'}:
            archive.write(path, str(path.relative_to(day7_root)))
print('DEVELOPMENT_FREEZE_SHA256', hashlib.sha256(freeze_path7.read_bytes()).hexdigest())
for model in freeze7['models']:
    print(json.dumps({k: model.get(k) for k in ('arm','seed','epoch','dev_utility','checkpoint_sha256')}))
print('TEST REMAINS UNOPENED. Commit this exact freeze before proceeding.')
files.download(str(dev_zip7))


In [ ]:
# Held-out access begins only after the actual checkpoint freeze is committed.
import subprocess, sys, os, json, hashlib
from pathlib import Path
freeze_commit7 = 'b603bd76ee97673beff91a79f08292d19276d85c'
freeze_git_path7 = 'experiments/day7/run_2026-10-09/checkpoint_freeze.json'
subprocess.run(['git','-C',str(project),'fetch','origin','experiment/part-consistency-colab'],check=True)
committed_freeze7 = subprocess.check_output(['git','-C',str(project),'show',freeze_commit7+':'+freeze_git_path7])
assert committed_freeze7 == (day7_root/'checkpoint_freeze.json').read_bytes()
assert hashlib.sha256(committed_freeze7).hexdigest() == 'd22db32b2fe60d648170ed38970b6dcb274f322cc5630b020db6d996f319dd62'
subprocess.run(['git','-C',str(project),'checkout','--detach',freeze_commit7],check=True)
print('COMMITTED_FREEZE_VERIFIED',freeze_commit7,flush=True)
cmd_test_extract7=[sys.executable,'-u',str(project/'experiments/day7/extract.py'),'--out',str(day7_features),'--upstream','/content/vjepa2','--splits','test','--checkpoint-freeze',str(day7_root/'checkpoint_freeze.json')]
with (day7_root/'extraction_test.log').open('w') as log:
    proc_test_extract7=subprocess.Popen(cmd_test_extract7,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,env=train_env)
    for line in proc_test_extract7.stdout:
        print(line,end='',flush=True);log.write(line);log.flush()
    test_extract_returncode7=proc_test_extract7.wait()
assert test_extract_returncode7==0, f'Test extraction failed: {test_extract_returncode7}'
manifest7=json.loads((day7_features/'manifest.json').read_text())
assert len(manifest7['clips'])==63
print('DAY7_HELD_OUT_FEATURES_READY',len(manifest7['clips']),flush=True)


In [ ]:
# Evaluate every frozen model, audit predictions, then intervene on frozen heads.
assert test_extract_returncode7 == 0
from datetime import datetime, timezone
(day7_root/'git_freeze_provenance.json').write_text(json.dumps({'freeze_commit':freeze_commit7,'freeze_repository_path':freeze_git_path7,'freeze_sha256':hashlib.sha256(committed_freeze7).hexdigest(),'exact_git_bytes_verified_before_test_extraction':True,'recorded_after_extraction_utc':datetime.now(timezone.utc).isoformat()},indent=2)+'\n')
def run_day7_stage7(stage,args):
    print('DAY7_STAGE_START',stage,flush=True)
    with (day7_root/(stage+'.log')).open('w') as log:
        proc=subprocess.Popen([sys.executable,'-u',*args],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,env=train_env)
        for line in proc.stdout:
            print(line,end='',flush=True);log.write(line);log.flush()
        code=proc.wait()
    assert code==0, f'{stage} failed: {code}'
    print('DAY7_STAGE_COMPLETE',stage,flush=True)
run_day7_stage7('evaluation',[str(project/'experiments/day7/train.py'),'--stage','test','--features',str(day7_features),'--out',str(day7_root),'--frozen-path',str(day7_root/'checkpoint_freeze.json')])
run_day7_stage7('independent_analysis',[str(project/'experiments/day7/analyze.py'),'--run',str(day7_root),'--out',str(day7_root/'analysis')])
run_day7_stage7('mechanism_diagnostics',[str(project/'experiments/day7/diagnose.py'),'--run',str(day7_root),'--features',str(day7_features)])
print('DAY7_TEST_AND_DIAGNOSTICS_COMPLETE',flush=True)


In [ ]:
# Preserve full learned heads/raw predictions and compact Git evidence separately.
assert (day7_root/'diagnostics/mechanism_reliance_v1/reliance_results.json').is_file()
day7_export = Path('/content/day7_export')
subprocess.run([sys.executable,'-u',str(project/'experiments/day7/export_artifacts.py'),'--run',str(day7_root),'--out',str(day7_export),'--features',str(day7_features)],check=True,env=train_env)
compact_zip7=Path('/content/Day7_Compact_Evidence.zip')
with zipfile.ZipFile(compact_zip7,'w',zipfile.ZIP_DEFLATED) as bundle:
    for path in sorted((day7_export/'compact').rglob('*')):
        if path.is_file(): bundle.write(path,str(path.relative_to(day7_export/'compact')))
    for name in ['export_summary.json','full_manifest.json']:
        bundle.write(day7_export/name,name)
print('DAY7_FULL_ARCHIVE_READY',str(day7_export/'full_run.zip'),flush=True)
print('DAY7_COMPACT_EVIDENCE_READY',compact_zip7.stat().st_size,hashlib.sha256(compact_zip7.read_bytes()).hexdigest(),flush=True)
files.download(str(compact_zip7))


In [ ]:
# Download the full original run, including all nine learned heads.
full_archive7=day7_export/'full_run.zip'
export_summary7=json.loads((day7_export/'export_summary.json').read_text())
assert hashlib.sha256(full_archive7.read_bytes()).hexdigest()==export_summary7['full_archive_sha256']
print('VERIFIED_FULL_ARCHIVE',full_archive7.stat().st_size,export_summary7['full_archive_sha256'])
files.download(str(full_archive7))


In [ ]:
# Audited result card: all test clips and all predetermined seeds.
from IPython.display import display, HTML
import pandas as pd
audit7=json.loads((day7_root/'analysis/independent_analysis.json').read_text())
assert audit7['status']=='passed'
cols7={'localization_accuracy_given_visible':'Visible part localization ↑','false_presence_given_absent':'False presence ↓','recovery_accuracy':'Immediate recovery ↑'}
rows7=[]
for method,label in [('global_full','Frozen JEPA matching'),('retrieval_only','Trained part memory'),('predictive','Part memory + predictive loss'),('coordinate_only','Position-only control')]:
    if method in audit7['seed_summary']:
        row={label2:f"{100*audit7['seed_summary'][method][metric]['mean']:.2f}%" for metric,label2 in cols7.items()}
    else:
        row={label2:f"{100*audit7['methods'][method]['overall'][metric]['rate']:.2f}%" for metric,label2 in cols7.items()}
    rows7.append({'Method':label,**row})
contrast7=audit7['primary_contrast']['balanced_utility']
lo7,hi7=[100*x for x in contrast7['clip_bootstrap_95ci']]
card7='<div style="font:15px Arial;padding:24px;max-width:960px;border:1px solid #ccd5de;border-radius:12px;background:#f7fafc;color:#17283b"><h2 style="margin-top:0">Day 7 · Trained JEPA part identity</h2><p>18 unseen synthetic videos · 3 training seeds · 871,810 trained parameters · No SAM</p>'+pd.DataFrame(rows7).to_html(index=False,border=0)+'<p><b>Predictive objective: no demonstrated tracking gain.</b> Balanced-score difference '+f"{100*contrast7['mean_difference']:+.3f} percentage points; paired 95% interval [{lo7:+.3f}, {hi7:+.3f}]."+'</p><p>Future-feature prediction improved (cosine 0.866 versus copying the initial part: 0.735), but identity recovery after occlusion remains unreliable.</p><p><b>What is built:</b> persistent part memory, same-car context, learned visibility and a future forecast used by tracking.<br><b>Scope:</b> synthetic patch-level tracking over a frozen encoder; real-video faces and editable masks remain untested.</p><p>All 26,784 prediction rows independently checked. All nine learned heads and original predictions archived.</p></div>'
display(HTML(card7))
